# Prepare COCO on Google Drive for Kaggle

Build a Kaggle-ready COCO directory from raw IDID labels and images on Google Drive.

Kaggle dataset must contain real image files, so a COCO JSON with paths to private Google Drive is not enough. Validation runs in `/content`, and only reports are copied back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Change these paths for your Drive layout.
PROJECT_DIR = Path('/content/drive/MyDrive/vlm-for-insulator-defect-detection')
INPUT_JSON = Path('/content/drive/MyDrive/idid_full/labels_v1.2.json')
IMAGES_DIR = Path('/content/drive/MyDrive/idid_full/images')

# Final Kaggle-ready dataset folder on Drive.
KAGGLE_COCO_DIR = Path('/content/drive/MyDrive/kaggle_datasets/idid_coco_v1')

# Validation runs in Colab local storage to avoid a second permanent copy on Drive.
VALIDATION_TMP_DIR = Path('/content/idid_coco_validated_tmp')

# Optional zip archive for manual Kaggle upload.
ZIP_BASE = Path('/content/drive/MyDrive/kaggle_datasets_archives/idid_coco_v1')

VAL_RATIO = 0.2
SEED = 42

print('PROJECT_DIR      =', PROJECT_DIR)
print('INPUT_JSON       =', INPUT_JSON)
print('IMAGES_DIR       =', IMAGES_DIR)
print('KAGGLE_COCO_DIR  =', KAGGLE_COCO_DIR)
print('VALIDATION_TMP   =', VALIDATION_TMP_DIR)
print('ZIP_BASE         =', ZIP_BASE)

In [ ]:
assert PROJECT_DIR.exists(), f'Missing repo directory: {PROJECT_DIR}'
assert INPUT_JSON.exists(), f'Missing labels JSON: {INPUT_JSON}'
assert IMAGES_DIR.exists(), f'Missing images directory: {IMAGES_DIR}'
assert (PROJECT_DIR / 'scripts' / 'idid_to_coco.py').exists(), 'scripts/idid_to_coco.py not found'
assert (PROJECT_DIR / 'scripts' / 'prepare_data.py').exists(), 'scripts/prepare_data.py not found'
print('All required paths exist.')

In [ ]:
!python -m pip install -q pillow

In [ ]:
import shutil
import subprocess
import sys

if KAGGLE_COCO_DIR.exists():
    shutil.rmtree(KAGGLE_COCO_DIR)

cmd = [
    sys.executable,
    'scripts/idid_to_coco.py',
    '--input-json', str(INPUT_JSON),
    '--images-dir', str(IMAGES_DIR),
    '--out-dir', str(KAGGLE_COCO_DIR),
    '--val-ratio', str(VAL_RATIO),
    '--seed', str(SEED),
    '--copy-images',
    '--summary-path', str(KAGGLE_COCO_DIR / 'reports' / 'conversion_summary.json'),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_DIR)

The Drive folder is now Kaggle-ready: it already contains split folders with real images and `annotations.json`.

Next, the validator runs in temporary Colab storage, and only reports are copied back to Drive.

In [ ]:
import shutil
import subprocess
import sys

if VALIDATION_TMP_DIR.exists():
    shutil.rmtree(VALIDATION_TMP_DIR)

cmd = [
    sys.executable,
    'scripts/prepare_data.py',
    '--raw_dir', str(KAGGLE_COCO_DIR),
    '--out_dir', str(VALIDATION_TMP_DIR),
    '--dataset', 'coco',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_DIR)

reports_src = VALIDATION_TMP_DIR / 'reports'
reports_dst = KAGGLE_COCO_DIR / 'reports'
reports_dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(reports_src, reports_dst, dirs_exist_ok=True)
shutil.rmtree(VALIDATION_TMP_DIR)

print('Reports copied to:', reports_dst)

In [ ]:
import json

summary_path = KAGGLE_COCO_DIR / 'reports' / 'conversion_summary.json'
validation_path = KAGGLE_COCO_DIR / 'reports' / 'validation_report.json'

with summary_path.open('r', encoding='utf-8') as f:
    summary = json.load(f)

with validation_path.open('r', encoding='utf-8') as f:
    validation = json.load(f)

print('Train images      :', summary['splits']['train']['num_images'])
print('Train annotations :', summary['splits']['train']['num_annotations'])
print('Val images        :', summary['splits']['val']['num_images'])
print('Val annotations   :', summary['splits']['val']['num_annotations'])
print('Validation status :', validation['status'])

In [ ]:
def folder_size_gb(path: Path) -> float:
    total = 0
    for p in path.rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 3)

print(f'KAGGLE_COCO_DIR size: {folder_size_gb(KAGGLE_COCO_DIR):.2f} GB')
for child in sorted(KAGGLE_COCO_DIR.iterdir()):
    print(child)

In [ ]:
import shutil

ZIP_BASE.parent.mkdir(parents=True, exist_ok=True)
zip_path = shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=KAGGLE_COCO_DIR)
print('Created zip archive:', zip_path)

## Optional Kaggle upload

If you upload from Colab with Kaggle CLI, put `kaggle.json` into `/content/.kaggle/kaggle.json`, create `dataset-metadata.json` inside `KAGGLE_COCO_DIR`, then use `kaggle datasets create -p <folder> --dir-mode zip` for first upload and `kaggle datasets version -p <folder> -m "update" --dir-mode zip` for updates.

If you use Kaggle web UI, upload the zip created above.

In [ ]:
import json

# Optional helper: create metadata for Kaggle CLI.
KAGGLE_USERNAME = 'your_kaggle_username'
KAGGLE_SLUG = 'idid-coco-v1'
KAGGLE_TITLE = 'IDID COCO v1'

metadata = {
    'title': KAGGLE_TITLE,
    'id': f'{KAGGLE_USERNAME}/{KAGGLE_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = KAGGLE_COCO_DIR / 'dataset-metadata.json'
with metadata_path.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Wrote:', metadata_path)
print('Edit KAGGLE_USERNAME, KAGGLE_SLUG, and KAGGLE_TITLE before upload.')